In [7]:
# Stwórz model Student z polami: id (Integer, primary key), name (String 100), age
# (Integer), email (String 150, unique). Utwórz tabelę w bazie.
# Wymagania:
# Definicja modelu
# Base.metadata.create_all()
# Sprawdzenie struktury tabel

from sqlalchemy import Column, Integer, String, Float, create_engine
from sqlalchemy.orm import declarative_base

Base = declarative_base()

class Student(Base):
    __tablename__ = 'students'

    id = Column(Integer, primary_key=True, autoincrement=True)
    name = Column(String(100), nullable=False)
    age = Column(Integer, default=0)
    email = Column(String(150), unique=True)

    def __repr__(self):
        return f"Student(id={self.id}, name='{self.name}', age={self.age}, email={self.email})"
    
engine = create_engine('sqlite:///students.db', echo=False)

# Tworzenie wszystkich tabel zdefiniowanych w Base
Base.metadata.create_all(engine)
print("✅ Tabela 'products' została utworzona!")
# Sprawdzenie struktury
from sqlalchemy import inspect
inspector = inspect(engine)
print("Tabele w bazie:", inspector.get_table_names())
print("Kolumny w 'students':", [col['name'] for col in inspector.get_columns('students')])

✅ Tabela 'products' została utworzona!
Tabele w bazie: ['students']
Kolumny w 'students': ['id', 'name', 'age', 'email']


In [ ]:
# Stwórz modele Movie i Actor z relacją Many-to-Many (aktorzy grają w wielu filmach, filmy
# mają wielu aktorów). Dodaj 5 filmów i 7 aktorów, przypisz aktorów do filmów.
# Dataset: Wykorzystaj prawdziwe nazwy filmów i aktorów (np. "Inception" - "Leonardo DiCaprio")
# Wymagania:
# Tabela asocjacyjna
# Dodanie danych z relacjami
# Zapytanie: "Jakie filmy ma aktor X?"
# Zapytanie: "Jacy aktorzy grają w filmie Y?"

from sqlalchemy import Column, Integer, String, ForeignKey, Table, create_engine
from sqlalchemy.orm import relationship, declarative_base

Base = declarative_base()

actor_movie_association = Table(
    'actor_movie',
    Base.metadata,
    Column('actor_id', Integer, ForeignKey('actors.id'), primary_key=True),
    Column('movie_id', Integer, ForeignKey('movies.id'), primary_key=True))
# Model Actor
class Actor(Base):
    __tablename__ = 'actors'

    id = Column(Integer, primary_key=True)
    name = Column(String(100), nullable=False)

# Relacja many to many
    movies = relationship("Movie", secondary=actor_movie_association, back_populates="actors")

    def __repr__(self):
        return f"<Actor(name='{self.name}', movies={len(self.movies)})>"
# Model Movie
class Movie(Base):
    __tablename__ = 'movies'

    id = Column(Integer, primary_key=True)
    title = Column(String(150), nullable=True)

    actors = relationship("Actor", secondary=actor_movie_association, back_populates="movies")

    def __repr__(self):
        return f"<Movie(title='{self.title}', actors={len(self.actors)})>"

# Utworzenie bazy
engine = create_engine('sqlite:///university.db', echo=False)
Base.metadata.create_all(engine)

Session = sessionmaker(bind=engine)
session = Session()

# Dodawanie danych

actor1 = Actor(name="Leonardo DiCaprio")
actor2 = Actor(name="Joseph Gordon-Levitt")
actor3 = Actor(name="Elliot Page")
actor4 = Actor(name="Keanu Reeves")
actor5 = Actor(name="Laurence Fishburne")
actor6 = Actor(name="Tom Hardy")
actor7 = Actor(name="Scarlett Johansson")

movie1 = Movie(title="Inception")
movie2 = Movie(title="The Matrix")
movie3 = Movie(title="The Matrix Reloaded")
movie4 = Movie(title="The Dark Knight Rises")
movie5 = Movie(title="Lucy")

# Przypisanie relacji

movie1.actors.extend([actor1, actor2, actor3, actor6])
movie2.actors.extend([actor4, actor5])
movie3.actors.extend([actor4, actor5])
movie4.actors.extend([actor6, actor1])
movie5.actors.append(actor7)

session.add_all([actor1, actor2, actor3, actor4, actor5, actor6, actor7, movie1, movie2, movie3, movie4, movie5])
session.commit()


# Zapytania po relacji
aktor = session.query(Actor).filter_by(name="Leonardo DiCaprio").first()
print(f"{aktor.name} gra w filmie: {[c.title for c in aktor.movies]}")

film = session.query(Movie).filter_by(title="Inception").first()
print(f"W filmie {film.title} gra {[c.name for c in film.actors]}")

Leonardo DiCaprio gra w filmie: ['Inception', 'The Dark Knight Rises']
W filmie Inception gra ['Leonardo DiCaprio', 'Elliot Page', 'Joseph Gordon-Levitt', 'Tom Hardy']


In [33]:
# Zainstaluj Alembic, zainicjalizuj w projekcie. Stwórz model Article (id, title, content).
# Wygeneruj migrację i zastosuj ją. Następnie dodaj kolumnę published_date i stwórz kolejną migrację.
# # Wymagania:
# alembic init , konfiguracja
# alembic revision --autogenerate
# alembic upgrade head
# Dodanie kolumny i ponowna migracja

from sqlalchemy import Column, Integer, String, Text
from sqlalchemy.orm import declarative_base

Base = declarative_base()

class Article(Base):
    __tablename__ = 'articles'

    id = Column(Integer, primary_key=True)
    title = Column(String(150), nullable=False)
    content = Column(Text, nullable=False)
    published_date = Column(Text, nullable=True) # Nowa kolumna

def upgrade():
    op.add_column('articles',
                  sa.Column('published_date', sa.Text(), nullable=True)
                  )
def downgrade():
    op.drop_column('articles', 
                   'published_date')